# 环节 10 · 输入与产物通道演示

纯 Python 标准库，零依赖。手搓四件事：

1. 路径穿越检测（`realpath` + 前缀校验）；
2. 归档条目检查（路径穿越 + 符号链接逃逸）；
3. zip bomb 判定（解压体积 + 嵌套层数）；
4. 产物回传四道校验（路径 / 类型 / 体积 / 内容）。

> 这两条通道是「盒子」上仅有的两个洞。

In [ ]:
# §1 路径穿越检测：必须在 realpath 之后比对前缀
import os
import tempfile

BASE = tempfile.mkdtemp(prefix="sbx-")
os.makedirs(os.path.join(BASE, "out"), exist_ok=True)
os.makedirs(os.path.join(BASE, "secret"), exist_ok=True)
with open(os.path.join(BASE, "secret", "key.txt"), "w") as f:
    f.write("SECRET")


def safe_join(base, user_path):
    base = os.path.realpath(base)
    target = os.path.realpath(os.path.join(base, user_path))
    if target != base and not target.startswith(base + os.sep):
        raise ValueError(f"path traversal: {user_path!r}")
    return target


CASES = ["out/report.txt", "../secret/key.txt", "out/../../secret/key.txt",
         "/etc/passwd", "out/./ok.txt"]
for p in CASES:
    try:
        print(f"允许  {p:<28} → {safe_join(os.path.join(BASE, 'out'), p)}")
    except ValueError as e:
        print(f"拒绝  {p:<28} → {e}")

# 符号链接组合攻击
link = os.path.join(BASE, "out", "link")
if not os.path.exists(link):
    os.symlink(os.path.join(BASE, "secret", "key.txt"), link)
try:
    print("\n符号链接:", safe_join(os.path.join(BASE, "out"), "link"))
    print("→ realpath 解析到了 BASE 之外，被前缀校验拦下 ✔")
except ValueError as e:
    print("\n符号链接:", e, "→ 被拦 ✔")
print("\n注意：只做 `if '..' in path` 的字符串检查，挡不住软链组合")

In [ ]:
# §2 归档条目检查：解包前逐条验证
def check_archive_entry(entry_name, target_dir):
    """返回 (是否安全, 原因)。只接受落在 target_dir 内的普通路径。"""
    if entry_name.startswith("/") or entry_name.startswith("\\"):
        return False, "绝对路径"
    if ".." in entry_name.split("/"):
        return False, "路径穿越"
    dest = os.path.normpath(os.path.join(target_dir, entry_name))
    if not dest.startswith(os.path.normpath(target_dir) + os.sep):
        return False, "越过目标目录"
    return True, "ok"


ENTRIES = [
    ("app/main.py",        True),
    ("app/../../etc/passwd", False),
    ("/etc/cron.d/evil",   False),
    ("app/../../../../root/.ssh/authorized_keys", False),
    ("app/data/x.csv",     True),
]
TARGET = "/tmp/sandbox/out"
for name, _ in ENTRIES:
    safe, why = check_archive_entry(name, TARGET)
    print(f"{'允许' if safe else '拒绝'}  {name:<46} {why}")
print()
print("还要额外拒的条目类型（本次未展开）：symlink / hardlink / 设备文件 / FIFO")
print("tar、zip、容器镜像层 都栽过同一类问题")

In [ ]:
# §3 压缩炸弹：压缩包大小完全不可信
MAX_RATIO = 100          # 允许的最大压缩比
MAX_TOTAL_MB = 2048      # 解压后总量上限
MAX_NEST = 3             # 最大嵌套层数


def check_bomb(compressed_mb, declared_uncompressed_mb, nest_level):
    ratio = declared_uncompressed_mb / max(compressed_mb, 1e-9)
    reasons = []
    if ratio > MAX_RATIO:
        reasons.append(f"压缩比 {ratio:.0f}x > {MAX_RATIO}x")
    if declared_uncompressed_mb > MAX_TOTAL_MB:
        reasons.append(f"解压后 {declared_uncompressed_mb} MB > {MAX_TOTAL_MB} MB")
    if nest_level > MAX_NEST:
        reasons.append(f"嵌套 {nest_level} 层 > {MAX_NEST} 层")
    return reasons


SAMPLES = [
    ("正常项目包",    10,    120,   1),
    ("臃肿的权重包",  800,   1600,  1),
    ("可疑高压缩比",  0.01,  200,   1),
    ("嵌套炸弹",      0.01,  300,   6),
]
for name, c, u, n in SAMPLES:
    reasons = check_bomb(c, u, n)
    print(f"{name:<14} 压缩 {c:>7} MB → 解压 {u:>6} MB  嵌套 {n}")
    print(f"{'  拒绝' if reasons else '  允许'}: {'; '.join(reasons) if reasons else 'ok'}")

print("\n关键三条：限『解压后总量』、限『嵌套层数』、限『压缩比』")
print("只限压缩包大小 = 没限")

In [ ]:
# §4 产物回传四道校验
ALLOWED_EXT = {".txt", ".json", ".csv", ".png", ".log"}
MAGIC = {b"\x89PNG": ".png", b"{": ".json", b"PK\x03\x04": ".zip"}


def sniff(data: bytes):
    for sig, ext in MAGIC.items():
        if data.startswith(sig):
            return ext
    return None


def validate_artifact(name, data_size_mb, head_bytes=b""):
    problems = []
    ext = os.path.splitext(name)[1].lower()
    if ext not in ALLOWED_EXT:
        problems.append(f"扩展名 {ext} 不在白名单")
    if data_size_mb > 100:
        problems.append(f"单文件 {data_size_mb} MB 超上限")
    if ".." in name or name.startswith("/"):
        problems.append("路径不安全")
    sniffed = sniff(head_bytes)
    if sniffed and ext == ".png" and sniffed != ".png":
        problems.append(f"魔数 {sniffed} 与扩展名不符")
    return problems


ARTIFACTS = [
    ("reports/summary.json", 2,   b'{"ok": true}'),
    ("images/chart.png",     1,   b"\x89PNG\r\n"),
    ("../../etc/passwd",     0.1, b"root:x:0:0"),
    ("payload.exe",          5,   b"MZ\x90\x00"),
    ("logs/run.log",         150, b"2026-09-12 INFO"),
    ("fake.png",             1,   b"PK\x03\x04"),
]
for name, size, head in ARTIFACTS:
    problems = validate_artifact(name, size, head)
    print(f"{'拒绝' if problems else '允许'}  {name:<24} {problems or 'ok'}")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 大仓库怎么进沙箱最稳？ | 只读挂载 + 沙箱内一次性可写层（读零拷贝、写不落宿主） |
| 2 | 路径穿越怎么防？ | `realpath` 解析后做前缀校验；字符串检查挡不住软链 |
| 3 | 解包归档要检查什么？ | 每个条目：目标路径仍在目录内 + 拒 symlink/hardlink/设备文件 |
| 4 | zip bomb 防大小还是防解压量？ | 解压后总量 + 嵌套层数 + 压缩比；压缩包大小不可信 |
| 5 | 为什么产物不走日志通道？ | 污染审计、打爆内存与超时；大对象走对象存储 + 受控凭证 |
| 6 | 依赖安装算「可信期」吗？ | 不算；`postinstall` / `setup.py` 就是任意代码执行 |

**相关长文**：[环节10-输入产物通道详解.md](./环节10-输入产物通道详解.md)